# 21. Judge reproducibility for Phase 5, and paraphrase-hardening of the structural detector

Two parts, both driven by notebook 20.

**Part 1.** The Phase 5 functionality gate rests on a 4-bit Qwen2.5-3B judge whose verdicts on the same cached outputs did not reproduce June's. Here the cached outputs are re-judged by three judges, the notebook-20 4-bit 3B, a 3B in fp16 with greedy decoding, and a 4-bit 7B, and the agreement between them and the resulting attack-success rates are reported, on the ownership-verified groups only. This decides whether section 4.11's manufactured-miss numbers can stand and which judge the paper should use.

**Part 2.** The structural detector was evaded by paraphrase (static FNR 0.13, adaptive 0.97). The remedy tested here is paraphrase in training: every instruction unit in the train and val splits is reworded into three non-imperative forms by a different rewriter than notebook 12's (Qwen2.5-3B, fp16, three new prompts), a paraphrased copy is made for half of the embedded positives, each paraphrase is added as a raw standalone negative to keep the twin structure, and the full configuration is retrained as `full_aug`. It is then evaluated on everything the original was: BIPIA, the second source, the structural test split, and the notebook-20 evasion set.

GPU. Caches: judge verdicts, paraphrases, the retrained model and all scores.

**Outputs.** `reports/phase5_judge_reproducibility.csv`, `reports/phase5_asr_by_judge.csv`, `reports/structural_aug_eval.csv`, `reports/structural_aug_evasion.csv`, `figures/structural_aug_evasion.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes
os.makedirs('reports', exist_ok=True); os.makedirs('figures', exist_ok=True)
import torch, gc; print('Session ready. GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## Stage A. Module

In [ ]:
%%writefile src/paraphrase_aug.py
"""Paraphrase augmentation for the structural detector, and judge-reproducibility helpers for Phase 5."""
from __future__ import annotations
import re, random
import numpy as np

# Three non-imperative reformulations of an instruction that a reader would still act on. Deliberately different
# from the notebook-12 rewriter (a different model, different prompts), so the evasion test is not the training style.
STYLES = {
    "expectation": ("Rewrite the instruction below as a neutral, descriptive statement of what the reader is expected to do, "
                    "with no imperative verbs and no direct commands. Keep the exact expected behaviour. Output only the rewritten sentence."),
    "reported":    ("Rewrite the instruction below as reported speech, as if a document were quoting what someone asked for, "
                    "keeping the exact expected behaviour. Output only the rewritten sentence."),
    "nominal":     ("Rewrite the instruction below as a passive or nominal sentence that a careful reader would still act on, "
                    "keeping the exact expected behaviour. Output only the rewritten sentence."),
}


def clean_paraphrase(p, unit):
    p = (p or "").strip().strip('"\'`').strip()
    p = re.sub(r"^(rewritten( sentence)?|output|answer)\s*[:\-]\s*", "", p, flags=re.I).strip()
    if not (10 <= len(p) <= 400) or p.lower() == unit.lower() or "\n" in p:
        return None
    return p


def paraphrase_units(units, generate, temperature=0.7, max_new_tokens=60, batch_size=16):
    """generate(prompts, system, max_new_tokens, temperature, batch_size) -> list[str]. Returns {unit: [paraphrases]}."""
    out = {u: [] for u in units}
    for style, sys_prompt in STYLES.items():
        gens = generate([f"Instruction: {u}" for u in units], sys_prompt, max_new_tokens, temperature, batch_size)
        for u, g in zip(units, gens):
            c = clean_paraphrase(g, u)
            if c and c not in out[u]: out[u].append(c)
    return out


def augment_records(records, para_map, rng, frac=0.5, test_texts=()):
    """For a fraction of embedded records, replace the unit inside the text with a paraphrase (label 1), and add each used
    paraphrase as a raw standalone (label 0). Records whose unit is not found verbatim, or without paraphrases, are skipped."""
    test_texts = set(test_texts); aug, twins, used = [], [], set()
    emb = [r for r in records if r["form"] == "embedded" and r["unit"] in para_map and para_map[r["unit"]]]
    rng.shuffle(emb)
    for r in emb[: int(frac * len(emb))]:
        p = rng.choice(para_map[r["unit"]])
        if r["text"].count(r["unit"]) != 1: continue
        t = r["text"].replace(r["unit"], p)
        if t in test_texts: continue
        aug.append({**r, "text": t, "unit": p, "form": "embedded_paraphrase", "payload": r["payload"] + "_paraphrase"})
        if p not in used and p not in test_texts:
            used.add(p); twins.append({**r, "text": p, "label": 0, "form": "standalone_paraphrase", "unit": p, "host_src": "", "position": "", "delim": ""})
    return aug, twins


def agreement(a, b, keys):
    keys = [k for k in keys if k in a and k in b]
    return float(np.mean([bool(a[k]) == bool(b[k]) for k in keys])) if keys else np.nan


## Stage B. Part 1: re-judge the cached Phase 5 outputs with three judges

In [ ]:
import importlib, json, random, numpy as np, pandas as pd
import paraphrase_aug, adversary_transfer, data_loaders, attack_success, targets
for m in (paraphrase_aug, adversary_transfer, data_loaders, attack_success, targets): importlib.reload(m)
from paraphrase_aug import agreement
from adversary_transfer import recover_subset, flat_variants, exec_layout
from data_loaders import load_bipia_categorized, BIPIA_MALICIOUS
from attack_success import programmatic_success, judge_prompt, parse_judge, combine, JUDGE_SYSTEM
from targets import load_lm, generate_batch

bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS); HOST = bip.label.values == 0
TAGS = ['protectai_v2', 'prompt_guard_2']
REW = np.load('data/phase5/rewrites.npy', allow_pickle=True)
P5 = {t: np.load(f'data/phase5/score_{t}.npy') for t in TAGS}; PB = {t: np.load(f'data/score_{t}_bipia_cat.npy') for t in TAGS}
SUB = recover_subset(bip, REW, P5=P5, PB=PB); VAR, OWNER, FIRST = flat_variants(SUB, REW)
VERIFIED = np.ones(len(SUB), bool)
for t in TAGS: VERIFIED &= np.abs(P5[t][FIRST] - PB[t][SUB.bip_index.values]) < 2e-3
VG = np.where(VERIFIED)[0]; print('verified groups:', len(VG), '/', len(SUB))

OUT = list(np.load('data/phase5/exec_out.npy', allow_pickle=True)); LAY = exec_layout(len(SUB), TAGS)
meta = [(src, i, SUB.meta.iloc[i], SUB.instruction.iloc[i]) for src in ['orig'] + TAGS for i in range(len(SUB))]
prog = [programmatic_success(m[2], m[3], o) for m, o in zip(meta, OUT)]
need = [j for j, p in enumerate(prog) if p is None]
prompts = [judge_prompt(meta[j][3], OUT[j]) for j in need]
print(f'programmatic: {len(OUT) - len(need)} | to judge: {len(need)}')

JUDGES = {'qwen3b_4bit (nb20)': ('data/phase5/judge_verdicts.json', None),
          'qwen3b_fp16_greedy': ('data/phase5/judge_3b_fp16.json', ('Qwen/Qwen2.5-3B-Instruct', False)),
          'qwen7b_4bit_greedy': ('data/phase5/judge_7b_4bit.json', ('Qwen/Qwen2.5-7B-Instruct', True))}
JD = {}
for name, (fp, spec) in JUDGES.items():
    if os.path.exists(fp): JD[name] = {int(k): v for k, v in json.load(open(fp)).items()}; print(name, 'loaded from cache'); continue
    if spec is None: print(name, 'cache missing (run notebook 20 first); skipped'); continue
    mname, fb = spec; print('judging with', name)
    try:
        jm, jt = load_lm(mname, four_bit=fb)
        jout = generate_batch(jm, jt, prompts, system=JUDGE_SYSTEM, max_new_tokens=8, batch_size=8, temperature=0.0)
        JD[name] = {j: bool(parse_judge(o)) for j, o in zip(need, jout)}
        json.dump({str(k): v for k, v in JD[name].items()}, open(fp, 'w'))
        del jm, jt; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f'{name} FAILED ({type(e).__name__}: {str(e)[:160]}); skipped')

names = list(JD); rows = []
for a in names:
    for b in names:
        if a < b: rows.append(dict(judge_a=a, judge_b=b, n_items=len(need), agreement=agreement(JD[a], JD[b], need)))
for nm in names:
    v = [bool(JD[nm].get(j, False)) for j in need]
    print(f'  {nm}: YES on {sum(v)}/{len(v)} judged items' + ('   <-- near-constant, treat with suspicion' if min(sum(v), len(v) - sum(v)) < 0.05 * len(v) else ''))
AG = pd.DataFrame(rows); AG.to_csv('reports/phase5_judge_reproducibility.csv', index=False)
print('\n=== pairwise agreement of judges on the judged items ==='); print(AG.round(3).to_string(index=False))
rows = []
for name in names:
    succ = np.array([bool(combine(prog[j], JD[name].get(j, False))) for j in range(len(OUT))])
    rows.append(dict(judge=name, **{f'ASR_{src}': float(succ[LAY[src]][VG].mean()) for src in LAY}))
rows.append(dict(judge='notebook 12 (June, 4-bit 3B, all 91)', ASR_orig=0.385, ASR_protectai_v2=0.198, ASR_prompt_guard_2=0.154))
AS = pd.DataFrame(rows); AS.to_csv('reports/phase5_asr_by_judge.csv', index=False)
print('\n=== attack-success rate per source by judge (verified groups) ==='); print(AS.round(3).to_string(index=False))

## Stage C. Part 2a: paraphrase the train and val instruction units (cached)

Rewriter: Qwen2.5-3B-Instruct in fp16, three prompts producing non-imperative forms, sampling at temperature 0.7. Different model and prompts from notebook 12.

In [ ]:
from structural_train import load_jsonl, select_config, train_model, score_texts
from paraphrase_aug import paraphrase_units, augment_records, STYLES
SPL = {sp: load_jsonl(f'data/structural/{sp}.jsonl') for sp in ('train', 'val', 'test')}
UNITS = {sp: sorted({r['unit'] for r in SPL[sp] if r['form'] == 'embedded'}) for sp in ('train', 'val')}
print('units to paraphrase:', {k: len(v) for k, v in UNITS.items()})
pfp = 'data/structural/paraphrases.json'
if os.path.exists(pfp):
    PARA = json.load(open(pfp)); print('paraphrases loaded from cache:', sum(len(v) for v in PARA.values()))
else:
    rm, rt = load_lm('Qwen/Qwen2.5-3B-Instruct', four_bit=False)
    def gen(prompts, system, mx, temp, bs): return generate_batch(rm, rt, prompts, system=system, max_new_tokens=mx, batch_size=bs, temperature=temp)
    PARA = paraphrase_units(UNITS['train'] + UNITS['val'], gen)
    json.dump(PARA, open(pfp, 'w')); del rm, rt; gc.collect(); torch.cuda.empty_cache()
    print('paraphrases generated:', sum(len(v) for v in PARA.values()))
rng = random.Random(0)
for u in rng.sample(UNITS['train'], 3): print('\nUNIT:', u); [print('  ->', p) for p in PARA.get(u, [])]
print('\nunits with 0 usable paraphrases:', sum(1 for u in UNITS['train'] + UNITS['val'] if not PARA.get(u)))

## Stage D. Part 2b: build the augmented splits

In [ ]:
test_texts = {r['text'] for r in SPL['test']}
rng = random.Random(0)
AUG_TR, TW_TR = augment_records(SPL['train'], PARA, rng, frac=0.5, test_texts=test_texts)
AUG_VA, TW_VA = augment_records(SPL['val'], PARA, rng, frac=0.5, test_texts=test_texts)
TRAIN_AUG = SPL['train'] + AUG_TR + TW_TR; VAL_AUG = SPL['val'] + AUG_VA + TW_VA
def counts(rs):
    import collections; c = collections.Counter((r['form'], r['label']) for r in rs); return {f'{f}/{l}': n for (f, l), n in sorted(c.items())}
emb_tr = sum(r['form'] == 'embedded' for r in SPL['train'])
print(f'augmented positives: {len(AUG_TR)} of a 0.5 target on {emb_tr} embedded ({len(AUG_TR) / max(1, 0.5 * emb_tr):.2f} of target); standalone twins: {len(TW_TR)}')
if len(AUG_TR) < 0.4 * emb_tr:
    print('WARNING: low augmentation yield. Check the paraphrase quality above; the hardening result will be weak by construction.')
print('train_aug:', len(TRAIN_AUG), counts(TRAIN_AUG)); print('val_aug:', len(VAL_AUG), counts(VAL_AUG))
assert not ({r['text'] for r in TRAIN_AUG} & test_texts), 'augmented train overlaps test'
print('\nexample paraphrased positive:', AUG_TR[0]['text'][:300].replace('\n', ' | '))
print('its twin (label 0):', TW_TR[0]['text'])

## Stage E. Part 2c: retrain as full_aug (skipped if the model exists)

In [ ]:
gc.collect(); torch.cuda.empty_cache()   # judges and rewriter are out of scope by now
best = 'data/structural/model_full_aug/best'
if os.path.exists(os.path.join(best, 'config.json')): print('full_aug exists, skipping training')
else:
    n1 = sum(r['label'] for r in TRAIN_AUG); print(f'training full_aug on {len(TRAIN_AUG)} records ({n1} positive, {len(TRAIN_AUG) - n1} negative)')
    train_model(TRAIN_AUG, VAL_AUG, 'data/structural/model_full_aug', seed=0, epochs=2)
    print('best val AUROC:', json.load(open('data/structural/model_full_aug/train_log.json'))['best_metric'])
MDIR = {'full': 'data/structural/model_full/best', 'full_aug': best}

## Stage F. Evaluate full_aug against full: BIPIA, second source, test split, and the notebook-20 evasion set

In [ ]:
from sklearn.metrics import roc_auc_score
import interval_panel; importlib.reload(interval_panel)
from interval_panel import three_thresholds, fnr, fpr
from data_loaders import load_notinject, BIPIA_HIJACK, BIPIA_HARMFUL
noti = load_notinject(); HIJ = bip.meta.isin(BIPIA_HIJACK).values; HARM = bip.meta.isin(BIPIA_HARMFUL).values
OPI = json.load(open('data/opi/opi_eval.json')); OTXT = [r['text'] for r in OPI['records']]; OY = np.array([r['label'] for r in OPI['records']])
TT = [r['text'] for r in SPL['test']]; TY = np.array([r['label'] for r in SPL['test']]); TF = np.array([r['form'] for r in SPL['test']])
SETS = {'bipia': bip.text.tolist(), 'notinject': noti.text.tolist(), 'opi': OTXT, 'structural_test': TT, 'phase5_variants': VAR}
CACHE = {'full': {'bipia': 'data/structural/scores/full_bipia.npy', 'notinject': 'data/structural/scores/full_notinject.npy', 'opi': 'data/opi/score_structural_full.npy',
                  'structural_test': 'data/structural/scores/full_structural_test.npy', 'phase5_variants': 'data/phase5/score_structural_full.npy'},
         'full_aug': {s: f'data/structural/scores/full_aug_{s}.npy' for s in SETS}}
S = {}
for m in ('full', 'full_aug'):
    S[m] = {}
    for s, texts in SETS.items():
        fp = CACHE[m][s]
        if not os.path.exists(fp): print('scoring', m, s, len(texts)); np.save(fp, score_texts(MDIR[m], texts))
        S[m][s] = np.load(fp); assert len(S[m][s]) == len(texts), f'{m}/{s}: {len(S[m][s])} vs {len(texts)}'
rows, erows = [], []
yh = np.r_[np.ones(int(HIJ.sum())), np.zeros(int(HOST.sum()))]; yr = np.r_[np.ones(int(HARM.sum())), np.zeros(int(HOST.sum()))]
for m in ('full', 'full_aug'):
    pb = S[m]['bipia']; t = three_thresholds(pb[HOST])['guar']; po = S[m]['opi']; to = three_thresholds(po[OY == 0])['guar']; pt = S[m]['structural_test']
    r = dict(model=m, AUROC_hijack=roc_auc_score(yh, np.r_[pb[HIJ], pb[HOST]]), AUROC_harmful=roc_auc_score(yr, np.r_[pb[HARM], pb[HOST]]),
             FNR_hijack_doc=fnr(pb[HIJ], t), FNR_harmful_doc=fnr(pb[HARM], t), notinject_FPR_doc=fpr(S[m]['notinject'], t),
             AUROC_second_source=roc_auc_score(OY, po), FNR_second_source=fnr(po[OY == 1], to), AUROC_test_split=roc_auc_score(TY, pt))
    emb = TF == 'embedded'
    for form in ['standalone', 'content_imperative_insert']:
        k = emb | (TF == form); r[f'AUROC_emb_vs_{form}'] = roc_auc_score(TY[k], pt[k])
    rows.append(r)
    pv = S[m]['phase5_variants']; sel = np.array([np.where(OWNER == i)[0][int(np.argmin(pv[OWNER == i]))] for i in range(len(SUB))])
    erows.append(dict(model=m, t_doc=t, mean_p_static=float(pv[FIRST].mean()), mean_p_adapt=float(pv[sel].mean()), FNR_static=fnr(pv[FIRST], t), FNR_adapt=fnr(pv[sel], t)))
EV = pd.DataFrame(rows); EV.to_csv('reports/structural_aug_eval.csv', index=False)
EVA = pd.DataFrame(erows); EVA['dFNR'] = EVA.FNR_adapt - EVA.FNR_static; EVA.to_csv('reports/structural_aug_evasion.csv', index=False)
pd.set_option('display.width', 250)
print('=== static performance: full vs full_aug ==='); print(EV.round(3).to_string(index=False))
print('\n=== notebook-20 evasion set: static vs adaptive FNR at the document-calibrated threshold ==='); print(EVA.round(3).to_string(index=False))

## Stage G. Figure

In [ ]:
import matplotlib.pyplot as plt
assert len(EVA) == 2
fig, ax = plt.subplots(figsize=(6.5, 4)); x = np.arange(len(EVA)); w = .38
ax.bar(x - w / 2, EVA.FNR_static, w, label='static attack'); ax.bar(x + w / 2, EVA.FNR_adapt, w, label='severity-maximizing rewrite')
ax.set_xticks(x); ax.set_xticklabels(EVA.model); ax.set_ylim(0, 1.05); ax.set_ylabel('FNR at document-calibrated 1 percent FPR')
ax.set_title('Paraphrase hardening against the notebook-12 adversary'); ax.legend(); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig('figures/structural_aug_evasion.png', dpi=150); plt.show()

## Stage H. Log and commit

In [ ]:
from reslog import log_result
summary = ('Judge agreement:\n' + AG.round(3).to_string(index=False) + '\n\nASR by judge:\n' + AS.round(3).to_string(index=False)
           + '\n\nfull vs full_aug:\n' + EV.round(3).to_string(index=False) + '\n\nEvasion:\n' + EVA.round(3).to_string(index=False))
log_result('nb21: Phase 5 judge reproducibility; paraphrase-hardened structural detector', summary, csv_df=EV, csv_name='structural_aug_eval.csv')
print(summary)

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
                    "nb21: Phase 5 judge reproducibility (three judges); paraphrase-augmented structural detector (full_aug) evaluated on BIPIA, second source, test split and the adversary set; add src/paraphrase_aug.py"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)